# Connecting to `ecommerce_dw` from Python

Sequential walkthrough: connect, read, write. Run each cell in order with `Shift+Enter`.

## Step 0 — Imports

In [12]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

## Step 1 — Build the connection URL

`URL.create` assembles the ODBC connection string piece by piece so we don't have to hand-escape special characters (like the `\` in the instance name).

- `Trusted_Connection=yes` logs in as the current Windows user — no password needed for local dev.
- `TrustServerCertificate=yes` is required with ODBC Driver 18: it encrypts connections by default and will refuse the local server's self-signed certificate otherwise.

In [13]:
connection_url = URL.create(
    "mssql+pyodbc",
    host=r"localhost\SQLEXPRESS",
    database="ecommerce_dw",
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "Trusted_Connection": "yes",
        "TrustServerCertificate": "yes",
    },
)

connection_url

mssql+pyodbc://localhost\SQLEXPRESS/ecommerce_dw?TrustServerCertificate=yes&Trusted_Connection=yes&driver=ODBC+Driver+18+for+SQL+Server

## Step 2 — Create the engine

The engine doesn't open a connection yet — it's a factory that hands out connections on demand and manages a small pool of them.

In [14]:
engine = create_engine(connection_url)

## Step 3 — Test the connection

`engine.connect()` opens an actual connection. `text(...)` marks a string as a literal SQL statement to run.

In [15]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION"))
    print(result.scalar())

Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (VM)



## Step 4 — Read data into a DataFrame

`pandas.read_sql` takes a query and an open connection, and returns the result as a DataFrame.

In [17]:
with engine.connect() as conn:
    df_customers = pd.read_sql("SELECT TOP 5 * FROM silver.customers", conn)

df_customers

,customer_id,first_name,last_name,email,phone,signup_date,city,state,country,profile_id,loyalty_tier,marketing_opt_in,preferred_channel,birth_date,gender,has_missing_value,has_invalid_value,has_outlier_value
0,CUST-000001,Bruce,Little,bruce.little90@hotmail.com,770-088-6008,2026-05-26,Amandaport,NC,United States,CRMP-001556,Bronze,True,Web,1988-12-21,Prefer not to say,False,False,False
1,CUST-000002,Andre,Li,andre.li433@hotmail.com,621-182-3398,2024-02-25,Lisaview,MI,United States,CRMP-002806,Silver,False,Web,1963-11-13,Female,False,False,False
2,CUST-000003,Laura,Miller,laura.miller202@yahoo.com,793-341-0166,2023-10-14,Tranton,TX,U.S.A.,CRMP-002565,Bronze,True,Marketplace,1953-04-18,nON-BINARY,False,False,False
3,CUST-000004,Dana,Warner,dana.warner735@gmail.com,765-918-0339,2024-11-06,Nelsonland,NC,USA,CRMP-000060,Silver,False,Web,2000-12-06,Male,False,False,False
4,CUST-000005,Gabrielle,Ramirez,gabrielle.ramirez513@hotmail.com,902-503-9261,2024-09-20,Franciscobury,TX,United States,CRMP-000295,Gold,False,Web,1968-09-24,Male,False,False,False


## Step 5 — Read with a parameter

Never build SQL by pasting a variable into an f-string — that's how SQL injection happens. Use `:name` placeholders and pass values through `params=` instead; SQLAlchemy sends them separately from the query text.

In [18]:
state = "CA"

with engine.connect() as conn:
    df_state = pd.read_sql(
        text("SELECT customer_id, first_name, last_name, city FROM silver.customers WHERE state = :state"),
        conn,
        params={"state": state},
    )

df_state

,customer_id,first_name,last_name,city
0,CUST-000035,Lisa,Mcneil,Derrickberg
1,CUST-000038,Melanie,Barrett,Rodriguezfort
2,CUST-000043,Kara,Marks,Gibsonstad
3,CUST-000063,Antonio,Guzman,Lake Nicole
4,CUST-000103,Pamela,Hart,Moranstad
...,...,...,...,...
229,CUST-002948,Rick,Morris,Clintonchester
230,CUST-002973,Michael,Pham,Mauricestad
231,CUST-002974,Morgan,Fisher,Port Jason
232,CUST-002981,Thomas,Walker,NaN


## Step 6 — Create a sandbox table to write into

We write to our own `dbo.python_write_demo` table rather than the pipeline's staging/bronze/silver/gold tables — those get truncated and rebuilt every time `ecommerce_pipeline.sql` runs, so writes from here would just get overwritten.

`engine.begin()` wraps the block in a transaction and commits automatically when the block finishes without error.

In [19]:
with engine.begin() as conn:
    conn.execute(text("""
        IF NOT EXISTS (
            SELECT 1 FROM sys.tables t JOIN sys.schemas s ON t.schema_id = s.schema_id
            WHERE s.name = 'dbo' AND t.name = 'python_write_demo'
        )
        CREATE TABLE dbo.python_write_demo (
            id INT IDENTITY(1,1) PRIMARY KEY,
            note NVARCHAR(200) NOT NULL,
            created_at DATETIME2 NOT NULL DEFAULT SYSUTCDATETIME()
        )
    """))

print("table ensured")

table ensured


## Step 7 — Insert a single row (parameterized)

In [20]:
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO dbo.python_write_demo (note) VALUES (:note)"),
        {"note": "hello from the notebook"},
    )

print("row inserted")

row inserted


## Step 8 — Bulk insert from a DataFrame

`DataFrame.to_sql` writes every row of a DataFrame to a table in one call — useful for query results, CSV loads, or anything already shaped like a table.

In [21]:
df_new_notes = pd.DataFrame({"note": ["notebook bulk row 1", "notebook bulk row 2"]})

df_new_notes.to_sql("python_write_demo", engine, schema="dbo", if_exists="append", index=False)

print("bulk rows inserted")

bulk rows inserted


## Step 9 — Read it back to confirm

In [22]:
with engine.connect() as conn:
    df_check = pd.read_sql("SELECT * FROM dbo.python_write_demo ORDER BY id", conn)

df_check

,id,note,created_at
0,1,hello from insert_one_row(),2026-09-09 17:46:57.374419
1,2,bulk row 1,2026-09-09 17:46:57.476832
2,3,bulk row 2,2026-09-09 17:46:57.476832
3,4,bulk row 3,2026-09-09 17:46:57.476832
4,5,hello from the notebook,2026-09-09 18:22:50.093430
5,6,notebook bulk row 1,2026-09-09 18:22:50.160720
6,7,notebook bulk row 2,2026-09-09 18:22:50.160720
7,8,hello from the notebook,2026-09-09 18:52:44.119832
8,9,notebook bulk row 1,2026-09-09 18:52:49.332990
9,10,notebook bulk row 2,2026-09-09 18:52:49.332990
